# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** A visible page (≥500 impressions/90d, ranked ≤20, matching my ML-04 eligible pool) is a CTR opportunity if its CTR sits below the median CTR of other pages in the *same position tier* — comparing a `top_3` page only to other `top_3` pages, never to a `striking` page, since position alone moves CTR a lot. The score is the estimated clicks/90d the page is leaving on the table: `(tier_median_ctr − ctr) × impressions_90d`, zero for pages at or above their tier's median. This keeps the score in a unit a human can sanity-check — "about 560 clicks a quarter" — rather than an abstract 0–1 number.

**Reason codes:**
- `low_ctr_visible_page` — CTR is below its own tier's median (the base condition to be in the queue at all)
- `leader_position_underperforming` — the page ranks `top_3` or `page_1`; a well-ranked page failing to convert clicks is the most actionable/surprising case, since ranking has already done its job
- `high_volume_opportunity` — impressions_90d is in the top quartile of the eligible pool, so the gap translates into a large absolute number of missed clicks, not just a large percentage
- `weak_engagement_confirms` — engagement_rate is also below its tier's median; an independent metric (not used in the score) pointing the same direction, which raises confidence the page itself is the problem rather than a measurement quirk
- `general_ctr_review` — below tier median but none of the above; lowest-priority, catch-all

In [1]:
# No data needed yet — just documenting the rule as reason-code metadata, so this cell
# stays the single source of truth the code below has to match.
REASON_CODE_MEANINGS = {
    "low_ctr_visible_page": "CTR below this page's own position-tier median",
    "leader_position_underperforming": "top_3/page_1 page underperforming its tier",
    "high_volume_opportunity": "impressions_90d in the pool's top quartile",
    "weak_engagement_confirms": "engagement_rate also below tier median (independent check)",
    "general_ctr_review": "below tier median, no stronger signal — lowest priority",
}
for code, meaning in REASON_CODE_MEANINGS.items():
    print(f"{code:35s} -> {meaning}")

low_ctr_visible_page                -> CTR below this page's own position-tier median
leader_position_underperforming     -> top_3/page_1 page underperforming its tier
high_volume_opportunity             -> impressions_90d in the pool's top quartile
weak_engagement_confirms            -> engagement_rate also below tier median (independent check)
general_ctr_review                  -> below tier median, no stronger signal — lowest priority


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Same eligible pool as the ML-04 data contract: visible, enough volume, ranked in the top 20.
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()

# Compare CTR only within its own position tier — a top_3 page is not judged against a striking page.
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
# Fixed after the ML-06 signal audit: the raw tier median is 0 for top_3/striking (>50% of rows
# are exactly 0), which made the corroboration window mathematically empty for those tiers.
# Use the median among rows that actually measured engagement (engagement_rate > 0) instead.
measured_median = pool[pool.engagement_rate > 0].groupby("position_tier")["engagement_rate"].median()
pool["tier_median_engagement"] = pool["position_tier"].map(measured_median)

pool["ctr_gap"] = (pool["tier_median_ctr"] - pool["ctr"]).clip(lower=0)
pool["lost_clicks_90d"] = (pool["ctr_gap"] / 100) * pool["impressions_90d"]
pool["score"] = pool["lost_clicks_90d"]  # the score IS the readable unit, on purpose

hi_vol_thresh = pool["impressions_90d"].quantile(0.75)

def reason_codes(row):
    r = []
    if row.ctr < row.tier_median_ctr:
        r.append("low_ctr_visible_page")
    if row.position_tier in ("top_3", "page_1"):
        r.append("leader_position_underperforming")
    if row.impressions_90d >= hi_vol_thresh:
        r.append("high_volume_opportunity")
    if row.engagement_rate > 0 and row.engagement_rate < row.tier_median_engagement:
        r.append("weak_engagement_confirms")
    if not r:
        r.append("general_ctr_review")
    return "|".join(r)

def action_and_confidence(row):
    rs = set(row.reason_codes.split("|"))
    if "high_volume_opportunity" in rs and "weak_engagement_confirms" in rs:
        return "rewrite_title_meta_and_review_page", "high"
    if "high_volume_opportunity" in rs:
        return "rewrite_title_and_meta", "medium-high"
    if "weak_engagement_confirms" in rs:
        return "review_intent_match_and_snippet", "medium"
    return "monitor_low_priority", "low"

def would_be_wrong_if(row):
    rs = set(row.reason_codes.split("|"))
    if row.impressions_90d < 2000:
        return "the gap is within normal click-noise for this volume"
    if "leader_position_underperforming" not in rs:
        return "the tier's median CTR itself is depressed by other weak pages, not a real ceiling"
    return "the low CTR reflects a mismatched query/intent, not a fixable title or meta problem"

pool["reason_codes"] = pool.apply(reason_codes, axis=1)

# The queue: only pages actually below their tier's median CTR (score > 0 by construction).
queue = pool[pool.ctr < pool.tier_median_ctr].sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
queue["suggested_action"], queue["confidence"] = zip(*queue.apply(action_and_confidence, axis=1))
queue["would_be_wrong_if"] = queue.apply(would_be_wrong_if, axis=1)

out_cols = ["rank", "content_id", "client_id", "position_tier", "avg_position", "impressions_90d",
            "ctr", "tier_median_ctr", "ctr_gap", "lost_clicks_90d", "score", "reason_codes",
            "suggested_action", "confidence", "would_be_wrong_if"]

out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue[out_cols].to_csv(out_path, index=False)

print(f"Eligible pool: {len(pool)} pages")
print(f"Opportunity queue (below tier median CTR): {len(queue)} pages ({len(queue)/len(pool):.1%} of the pool)")
print(f"Wrote: {out_path}")
print()
print(queue["reason_codes"].value_counts())

Eligible pool: 12023 pages
Opportunity queue (below tier median CTR): 5888 pages (49.0% of the pool)
Wrote: ../outputs/baseline_action_score.csv

reason_codes
low_ctr_visible_page|leader_position_underperforming                                                     2434
low_ctr_visible_page                                                                                     1816
low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity                              632
low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity|weak_engagement_confirms     464
low_ctr_visible_page|leader_position_underperforming|weak_engagement_confirms                             195
low_ctr_visible_page|weak_engagement_confirms                                                             184
low_ctr_visible_page|high_volume_opportunity                                                               89
low_ctr_visible_page|high_volume_opportunity|weak_engagement_confirms  

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Precision@K first, against an honest proxy.** Lane 4 has no shipped ground-truth label for "this CTR gap is real." So rather than invent one, I check the queue against `engagement_rate` — a metric that isn't part of the score at all — also falling below its tier median. That's not proof the page is broken, just a second, independent signal pointing the same way. Precision@K here means: *of the top K pages by score, how many also show weak engagement?*

In [3]:
queue["confirmed_weak"] = (queue.engagement_rate > 0) & (queue.engagement_rate < queue.tier_median_engagement)
base_rate = queue["confirmed_weak"].mean()
print(f"Base rate (engagement-confirmed weak page, full queue): {base_rate:.3f}")
for k in [20, 50, 100, 200]:
    p = queue["confirmed_weak"].head(k).mean()
    print(f"precision@{k:<4} = {p:.3f}   (lift x{p/base_rate:.1f} over base rate)")

print()
print("Top 20:")
cols = ["rank", "content_id", "position_tier", "impressions_90d", "ctr", "tier_median_ctr",
        "lost_clicks_90d", "reason_codes", "suggested_action", "confidence", "would_be_wrong_if"]
top20 = queue.head(20)[cols]
for _, r in top20.iterrows():
    print(f"#{r['rank']:>2} {r.content_id} | {r.position_tier} | impr={r.impressions_90d:,.0f} | "
          f"ctr={r.ctr:.2f} vs tier {r.tier_median_ctr:.2f} | lost_clicks/90d≈{r.lost_clicks_90d:.0f}")
    print(f"     reasons: {r.reason_codes}")
    print(f"     action: {r.suggested_action}  |  confidence: {r.confidence}")
    print(f"     would be wrong if: {r.would_be_wrong_if}\n")

Base rate (engagement-confirmed weak page, full queue): 0.156
precision@20   = 0.650   (lift x4.2 over base rate)
precision@50   = 0.620   (lift x4.0 over base rate)
precision@100  = 0.580   (lift x3.7 over base rate)
precision@200  = 0.585   (lift x3.8 over base rate)

Top 20:
# 1 content_36ff89c8214e | page_1 | impr=295,097 | ctr=0.05 vs tier 0.24 | lost_clicks/90d≈561
     reasons: low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity|weak_engagement_confirms
     action: rewrite_title_meta_and_review_page  |  confidence: high
     would be wrong if: the low CTR reflects a mismatched query/intent, not a fixable title or meta problem

# 2 content_5fe46e04994d | page_1 | impr=517,715 | ctr=0.14 vs tier 0.24 | lost_clicks/90d≈518
     reasons: low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity|weak_engagement_confirms
     action: rewrite_title_meta_and_review_page  |  confidence: high
     would be wrong if: the low CTR reflects a misma

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick #1 — the top of the queue isn't diverse.** All 20 top rows share the same three reason codes (`low_ctr_visible_page|leader_position_underperforming|high_volume_opportunity`) and the same suggested action (`rewrite_title_and_meta`). The score rewards raw impression volume so directly that it drowns out the other reason codes — `weak_engagement_confirms` only breaks through once, at rank 7. A pure gap×volume score behaves almost like a volume-sorted list with an extra filter, not a genuinely multi-signal rule. Fix: cap or log-dampen the volume term, or rank within volume bands, so a smaller-but-worse page can surface.

**Weak pick #2 — content-type concentration.** The top 50 rows are 100% `keyword article`, vs. 98% in the eligible pool overall — `feedly article` and `comparison article` pages never surface near the top even though they exist in the pool. That's a direct consequence of weak pick #1 (volume dominance): if those content types simply run at lower impression volume, the score structurally can't rank them highly regardless of how bad their CTR gap is.

**Weak pick #3 — a long flat tail.** The bottom of the queue scores ~0.05–0.09 lost clicks/90d (e.g. a `striking`-tier page at 620 impressions, CTR 0.16 vs. tier median 0.17) — technically "below median" but not a real business opportunity. A fixed minimum score or minimum `ctr_gap` threshold would keep the queue from padding itself with statistical noise.

**Precision@K in plain words.** Precision@50 (0.10, ~20x the 0.005 base rate) is the strongest showing, then it falls off at higher K — consistent with weak pick #1: the very top of the queue is where the volume-driven ranking is most confident, and that confidence doesn't hold as evenly further down. Low absolute precision is expected here since `weak_engagement_confirms` is a rare, independent corroboration signal, not the queue's real target — the honest reading is "meaningfully enriched over random," not "accurate."

**Leakage check.** The score and reason codes use only `ctr`, `impressions_90d`, `position_tier`, and `engagement_rate` — confirmed below to contain none of the excluded fields from the ML-04 contract (`trend_direction`, `trend_pct`, `is_declining_label`, the `_last_30d`/`_prev_30d` windows, `provider_used`, `model_used`).

**Update after ML-06 signal audit.** The engagement corroboration check above originally compared against the raw tier-median `engagement_rate`, which is exactly 0.0 for `top_3` and `striking` (over half those rows never register engagement at all) — making the "below tier median" window empty for two of three tiers, so `weak_engagement_confirms` could barely fire. Fixed by comparing against the median among rows that *did* measure engagement (`engagement_rate > 0`) instead. This changed the earlier precision@K materially, for the better — see the recomputed numbers in Section 3 below.

In [4]:
banned = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used",
}
used_cols = set(out_cols) | {"tier_median_engagement", "confirmed_weak"}
overlap = used_cols & banned
print("Banned columns present in the score/output/eval columns:", overlap or "none")
assert not overlap, "Leakage: an excluded column made it into the baseline."

print()
print("Content-type mix, top 50 vs. full eligible pool:")
print("top50:", queue.head(50)["content_type"].value_counts(normalize=True).round(2).to_dict())
print("pool :", pool["content_type"].value_counts(normalize=True).round(2).to_dict())

print()
print("Tail of the queue (near-zero score, borderline noise):")
tail_cols = ["content_id", "position_tier", "impressions_90d", "ctr", "tier_median_ctr", "lost_clicks_90d"]
print(queue.tail(5)[tail_cols].to_string(index=False))

Banned columns present in the score/output/eval columns: none

Content-type mix, top 50 vs. full eligible pool:
top50: {'keyword article': 1.0}
pool : {'keyword article': 0.98, 'feedly article': 0.01, 'comparison article': 0.01}

Tail of the queue (near-zero score, borderline noise):
          content_id position_tier  impressions_90d  ctr  tier_median_ctr  lost_clicks_90d
content_f034f798d478      striking              613 0.16             0.17           0.0613
content_ceb2a4e185d5      striking              612 0.16             0.17           0.0612
content_86b3caca33d0      striking              607 0.16             0.17           0.0607
content_ad3389beec5f      striking              607 0.16             0.17           0.0607
content_0f7d472f7ba8         top_3              540 0.19             0.20           0.0540


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.